# Website Editor

Run all cells top to bottom, edit the fields, then click **Save Changes** at the bottom.

Fields that say `(HTML)` contain raw HTML — links, bold/italic tags, and `<br>` tags are preserved as-is. Edit them like HTML source.

In [2]:
# Uncomment if beautifulsoup4 or ipywidgets are not installed:
# !pip install beautifulsoup4 ipywidgets

from bs4 import BeautifulSoup, NavigableString
import ipywidgets as widgets
from IPython.display import display
import pathlib

HTML_PATH = pathlib.Path('index.html')
soup = BeautifulSoup(HTML_PATH.read_text(), 'html.parser')

def set_inner(tag, html_str):
    """Replace a tag's inner content with parsed HTML."""
    new = BeautifulSoup(f'<div>{html_str}</div>', 'html.parser').div
    tag.clear()
    for child in list(new.children):
        tag.append(child)

print('index.html loaded ✓')

index.html loaded ✓


In [3]:
# --- Profile ---
_logo_a       = soup.select_one('#header header #logo a')
_tagline_p    = soup.select_one('#header header p')
_tagline_links = _tagline_p.find_all('a')
_scholar_a, _cv_a = _tagline_links[0], _tagline_links[1]

w_name    = widgets.Text(value=_logo_a.get_text(),                         description='Name:',       layout=widgets.Layout(width='400px'))
w_tagline = widgets.Text(value=str(_tagline_p.contents[0]).strip(),        description='Tagline:',    layout=widgets.Layout(width='550px'))
w_scholar = widgets.Text(value=_scholar_a['href'],                         description='Scholar URL:', layout=widgets.Layout(width='650px'))
w_cv      = widgets.Text(value=_cv_a['href'],                              description='CV URL:',     layout=widgets.Layout(width='650px'))

display(widgets.VBox([
    widgets.HTML('<h3>Profile</h3>'),
    w_name, w_tagline, w_scholar, w_cv
]))

In [4]:
# --- About ---
_about_h2      = soup.select_one('#about .container header.major h2')
_about_intro_p = soup.select_one('#about .container header.major p')
_about_bio_p   = soup.select_one('#about .container > p')

w_about_intro = widgets.Textarea(
    value=_about_intro_p.decode_contents(),
    description='Intro (HTML):',
    layout=widgets.Layout(width='700px', height='80px')
)
w_about_bio = widgets.Textarea(
    value=_about_bio_p.decode_contents(),
    description='Bio (HTML):',
    layout=widgets.Layout(width='700px', height='220px')
)

display(widgets.VBox([
    widgets.HTML('<h3>About</h3>'),
    w_about_intro,
    w_about_bio
]))

In [5]:
# --- Publications ---
_pub_articles = soup.select('#publications .features article')
pub_ws = []

display(widgets.HTML('<h3>Publications</h3>'))

for i, art in enumerate(_pub_articles):
    wt = widgets.Text(
        value=art.select_one('.inner h4').get_text(),
        description='Title:',
        layout=widgets.Layout(width='650px')
    )
    wl = widgets.Text(
        value=art.select_one('a.image')['href'],
        description='Link (DOI):',
        layout=widgets.Layout(width='650px')
    )
    wc = widgets.Textarea(
        value=art.select_one('.inner p').decode_contents(),
        description='Citation (HTML):',
        layout=widgets.Layout(width='700px', height='120px')
    )
    pub_ws.append({'title': wt, 'link': wl, 'citation': wc})
    display(widgets.VBox([
        widgets.HTML(f'<b>Publication {i + 1}</b>'),
        wt, wl, wc
    ]))

HTML(value='<h3>Publications</h3>')

In [6]:
# --- Other Projects ---
_proj_articles = soup.select('#other .features article')
proj_ws = []

display(widgets.HTML('<h3>Other Projects</h3>'))

for i, art in enumerate(_proj_articles):
    wt = widgets.Text(
        value=art.select_one('.inner h4').get_text(),
        description='Title:',
        layout=widgets.Layout(width='650px')
    )
    wl = widgets.Text(
        value=art.select_one('a.image')['href'],
        description='Link:',
        layout=widgets.Layout(width='650px')
    )
    wd = widgets.Textarea(
        value=art.select_one('.inner p').decode_contents(),
        description='Description:',
        layout=widgets.Layout(width='700px', height='100px')
    )
    proj_ws.append({'title': wt, 'link': wl, 'description': wd})
    display(widgets.VBox([
        widgets.HTML(f'<b>Project {i + 1}</b>'),
        wt, wl, wd
    ]))

HTML(value='<h3>Other Projects</h3>')

In [7]:
# --- Social Links ---
_social_links = soup.select('#header footer .icons li a')

def _find_social(icon_class):
    return next(a for a in _social_links if icon_class in ' '.join(a.get('class', [])))

_twitter_a = _find_social('fa-twitter')
_github_a  = _find_social('fa-github')
_email_a   = _find_social('fa-envelope')

w_twitter = widgets.Text(value=_twitter_a['href'],                                         description='Twitter URL:', layout=widgets.Layout(width='450px'))
w_github  = widgets.Text(value=_github_a['href'],                                          description='GitHub URL:',  layout=widgets.Layout(width='450px'))
w_email   = widgets.Text(value=_email_a['href'].replace('mailto:', '').rstrip('?'),        description='Email:',       layout=widgets.Layout(width='450px'))

display(widgets.VBox([
    widgets.HTML('<h3>Social Links</h3>'),
    w_twitter, w_github, w_email
]))

In [8]:
# --- Save ---
_save_out = widgets.Output()

def _save(_btn):
    # Profile
    _logo_a.string = w_name.value
    _tagline_p.contents[0].replace_with(NavigableString(w_tagline.value))
    _scholar_a['href'] = w_scholar.value
    _cv_a['href'] = w_cv.value

    # About (name appears in both header and about section)
    _about_h2.string = w_name.value
    set_inner(_about_intro_p, w_about_intro.value)
    set_inner(_about_bio_p, w_about_bio.value)

    # Publications
    for art, ws in zip(_pub_articles, pub_ws):
        art.select_one('.inner h4').string = ws['title'].value
        art.select_one('a.image')['href'] = ws['link'].value
        set_inner(art.select_one('.inner p'), ws['citation'].value)

    # Other Projects
    for art, ws in zip(_proj_articles, proj_ws):
        art.select_one('.inner h4').string = ws['title'].value
        art.select_one('a.image')['href'] = ws['link'].value
        set_inner(art.select_one('.inner p'), ws['description'].value)

    # Social links
    _twitter_a['href'] = w_twitter.value
    _github_a['href']  = w_github.value
    _email_a['href']   = f'mailto:{w_email.value}?'

    HTML_PATH.write_text(str(soup))
    _save_out.clear_output()
    with _save_out:
        print('Saved to index.html ✓')

_btn = widgets.Button(
    description='Save Changes',
    button_style='success',
    layout=widgets.Layout(width='200px', height='40px')
)
_btn.on_click(_save)

display(widgets.VBox([
    widgets.HTML('<h3>Save</h3>'),
    _btn,
    _save_out
]))